# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeref538/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks — test the rule's assumptions first

Two signals, one bucket table each with `n` printed, one verdict each. Both are signals behind real FlyRank flags from this week's session.


In [1]:
# SIGNAL CHECKS — test two signals BEFORE encoding any rule.
# Both are signals behind real FlyRank flags from the session:
#   Signal 1 = staleness (behind the refresh flags)
#   Signal 2 = CTR-vs-position (behind the "needs CTR fix" logic)
import os
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# label for checking signals only - NEVER a feature in the rule
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} pages | overall declining rate: {df['is_declining'].mean():.3f}\n")

print("=" * 68)
print("SIGNAL 1 - staleness (the signal behind FlyRank's refresh flags)")
print("Claim I want to test: the staler a page, the more likely it is declining.")
print("=" * 68)
df["stale_bucket"] = pd.cut(df["days_since_last_update"],
                            [-1, 30, 90, 180, 365, 10**9],
                            labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"])
t1 = df.groupby("stale_bucket", observed=True).agg(
    n=("is_declining", "size"),
    declining_rate=("is_declining", "mean")).round(3)
print(t1.to_string())
print(f"\nPages passing the standard 180-day refresh threshold: "
      f"{(df['days_since_last_update'] >= 180).sum()} of {len(df):,}")

30,000 pages | overall declining rate: 0.542

SIGNAL 1 - staleness (the signal behind FlyRank's refresh flags)
Claim I want to test: the staler a page, the more likely it is declining.
                  n  declining_rate
stale_bucket                       
0-30d         20480           0.511
31-90d          175           0.589
91-180d        9171           0.611
181-365d        169           0.467
365d+             5           0.600

Pages passing the standard 180-day refresh threshold: 174 of 30,000


**Signal 1 verdict: MIXED — and this saved my rule.**

The declining rate does climb with staleness at first (0.511 → 0.589 → 0.611 across 0–30d, 31–90d, 91–180d), which supports the idea. But it then **drops to 0.467 in the 181–365d bucket** — the group the standard 180-day refresh threshold actually targets. The 365d+ bucket looks high (0.600) but n=5, far too small to lean on.

So the naive reading ("staler = more likely declining") is not safe here. Worse for the rule: only **17 of 30,000 pages** even pass a 180-day threshold, so a hard `days_since_last_update >= 180` gate can't fill a review queue at all. **Decision this drove:** I use staleness as a *small additive tiebreaker*, never as a hard gate — exactly because this check came back MIXED rather than CONFIRMED.

In [2]:
print("=" * 68)
print("SIGNAL 2 - CTR vs position (the signal behind the 'needs CTR fix' logic)")
print("Claim A: CTR depends on position, so CTR is only comparable WITHIN a tier.")
print("Claim B: within a tier, below-median CTR means more likely declining.")
print("=" * 68)

visible = df[df["impressions_90d"] >= 100]
t2 = visible.groupby("position_tier", observed=True).agg(
    n=("ctr", "size"),
    mean_ctr=("ctr", "mean"),
    median_ctr=("ctr", "median")).round(3).sort_values("mean_ctr", ascending=False)
print("\nClaim A - mean CTR by position tier (impressions >= 100):")
print(t2.to_string())

print("\nClaim B - within each tier, split at that tier's median CTR:")
rows = []
for tier in ["page_1", "striking", "page_3_5"]:
    sub = visible[visible["position_tier"] == tier].copy()
    below = sub["ctr"] < sub["ctr"].median()
    rows.append({"tier": tier,
                 "n_below": int(below.sum()),
                 "declining_below_median_ctr": round(sub.loc[below, "is_declining"].mean(), 3),
                 "n_above": int((~below).sum()),
                 "declining_above_median_ctr": round(sub.loc[~below, "is_declining"].mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False))


SIGNAL 2 - CTR vs position (the signal behind the 'needs CTR fix' logic)
Claim A: CTR depends on position, so CTR is only comparable WITHIN a tier.
Claim B: within a tier, below-median CTR means more likely declining.

Claim A - mean CTR by position tier (impressions >= 100):
                  n  mean_ctr  median_ctr
position_tier                            
page_1         8633     0.355        0.23
top_3           533     0.334        0.19
striking       5903     0.256        0.15
page_3_5       6058     0.142        0.06
deep            879     0.055        0.00

Claim B - within each tier, split at that tier's median CTR:
    tier  n_below  declining_below_median_ctr  n_above  declining_above_median_ctr
  page_1     4262                       0.680     4371                       0.535
striking     2867                       0.683     3036                       0.573
page_3_5     2914                       0.588     3144                       0.580


**Signal 2 verdict: CONFIRMED (with one honest caveat).**

**Claim A is confirmed strongly.** Mean CTR falls steadily as position worsens: page_1 0.355 → top_3 0.334 → striking 0.256 → page_3_5 0.142 → deep 0.055. That's the CTR cliff, and it's why comparing raw CTR across pages is meaningless — a 0.15% CTR is bad on page 1 and good on page 3–5. Any CTR rule must compare a page to **its own tier**. (Small oddity: `top_3` sits slightly *below* `page_1` rather than above it, which I'd want to understand before making strong claims about the very top of the SERP.)

**Claim B is confirmed for good positions only.** Splitting each tier at its own median CTR:
- `page_1`: below-median CTR → 0.680 declining vs 0.535 above (a real 14-point gap)
- `striking`: 0.683 vs 0.573 (11-point gap)
- `page_3_5`: 0.588 vs 0.580 — **essentially no difference**

So the signal is real where the page has something to lose, and vanishes in deep positions. **Decision this drove:** my rule gates on good positions (`avg_position <= 20`) and measures the CTR gap *within tier*, because that's the only place the signal actually holds.

## 2. My rule and its reason codes


**The rule, in plain words:** *A page is worth reviewing first if it already ranks well enough for people to see it, but earns fewer clicks than other pages at the same rank — and among equally-underperforming pages, prefer the one that hasn't been touched in longer.*

**Written as a score (no fitted weights, readable on purpose):**

```
ctr_gap_ratio  = (tier_median_ctr - page_ctr) / tier_median_ctr   , floored at 0
freshness_term = days_since_last_update / 365                     , capped at 1
score          = ctr_gap_ratio + 0.3 * freshness_term
```

The CTR gap is the main term because Signal 2 came back CONFIRMED. Freshness gets a small 0.3 weight — deliberately a tiebreaker, not a gate — because Signal 1 came back MIXED.

**Eligibility gate (who's even in the queue):** `impressions_90d >= 250` (enough exposure that a CTR ratio isn't noise) and `0 < avg_position <= 20` (the signal only held at good positions; `avg_position = 0` means "no data", not rank zero).

**Reason codes and their actions — one reason code per page:**

| Reason code | When | Action |
|---|---|---|
| `far_below_tier_ctr` | CTR gap ≥ 75% of tier median | rewrite title/meta |
| `below_tier_ctr` | CTR gap ≥ 25% | review snippet |
| `aging_but_ctr_ok` | CTR fine, but ≥ half a year old | refresh content |
| `no_clear_gap` | neither | monitor |

**No leakage:** the score uses only `ctr`, `avg_position`, `position_tier`, `impressions_90d`, and `days_since_last_update` — all knowable before any outcome. `trend_direction` and `trend_pct` are used *only* to score the rule afterwards, never as inputs.

In [3]:
# BUILD THE RANKED QUEUE and write the CSV.
import json

# --- eligibility gate ---
elig = df[(df["impressions_90d"] >= 250) &
          (df["avg_position"] > 0) &
          (df["avg_position"] <= 20)].copy()
print(f"Eligible pages: {len(elig):,} of {len(df):,}")

# --- the score ---
tier_median = elig.groupby("position_tier")["ctr"].transform("median")
elig["tier_median_ctr"] = tier_median
elig["ctr_gap_ratio"] = ((tier_median - elig["ctr"]).clip(lower=0)
                         / tier_median.replace(0, np.nan)).fillna(0)
elig["freshness_term"] = (elig["days_since_last_update"] / 365).clip(upper=1)
elig["baseline_action_score"] = (elig["ctr_gap_ratio"] + 0.3 * elig["freshness_term"]).round(4)

# --- one reason code + action per page ---
def reason_code(r):
    if r["ctr_gap_ratio"] >= 0.75:
        return "far_below_tier_ctr"
    if r["ctr_gap_ratio"] >= 0.25:
        return "below_tier_ctr"
    if r["freshness_term"] >= 0.5:
        return "aging_but_ctr_ok"
    return "no_clear_gap"

ACTIONS = {"far_below_tier_ctr": "rewrite title/meta",
           "below_tier_ctr": "review snippet",
           "aging_but_ctr_ok": "refresh content",
           "no_clear_gap": "monitor"}
elig["reason_code"] = elig.apply(reason_code, axis=1)
elig["action"] = elig["reason_code"].map(ACTIONS)

queue = elig.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
queue["queue_rank"] = queue.index + 1

# --- score the rule (label used ONLY here, for evaluation) ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = elig["is_declining"].values
s = elig["baseline_action_score"].values
base_rate = y.mean()
metrics = {"eligible_pages": int(len(elig)), "base_rate": round(float(base_rate), 3),
           "precision_at_10": round(float(precision_at_k(s, y, 10)), 3),
           "precision_at_20": round(float(precision_at_k(s, y, 20)), 3),
           "precision_at_50": round(float(precision_at_k(s, y, 50)), 3)}
print(f"\nBase rate (a random pick from the eligible slice): {base_rate:.3f}")
for k in (10, 20, 50):
    print(f"Baseline rule Precision@{k}: {metrics[f'precision_at_{k}']:.3f}")

# --- write the queue CSV (regenerated each run; git-ignored by the leak guard) ---
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["queue_rank", "content_id", "client_id", "baseline_action_score", "reason_code",
            "action", "impressions_90d", "avg_position", "position_tier", "ctr",
            "tier_median_ctr", "ctr_gap_ratio", "days_since_last_update"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nWrote work/outputs/baseline_action_score.csv ({len(queue):,} rows)")
print("Wrote work/outputs/w04_baseline_metrics.json (the run's receipts)")
print("\nAction mix across the whole queue:")
print(queue["action"].value_counts().to_string())

Eligible pages: 13,562 of 30,000

Base rate (a random pick from the eligible slice): 0.610
Baseline rule Precision@10: 0.900
Baseline rule Precision@20: 0.900
Baseline rule Precision@50: 0.800



Wrote work/outputs/baseline_action_score.csv (13,562 rows)
Wrote work/outputs/w04_baseline_metrics.json (the run's receipts)

Action mix across the whole queue:
action
monitor               8181
review snippet        2824
rewrite title/meta    2544
refresh content         13


## 3. Top-10 review


In [4]:
# TOP-10 REVIEW — read my own queue with a skeptic's eye.
show = ["queue_rank", "content_id", "baseline_action_score", "reason_code", "action",
        "impressions_90d", "avg_position", "position_tier", "ctr", "tier_median_ctr",
        "days_since_last_update", "is_declining"]
top10 = queue.head(10)
print(top10[show].to_string(index=False))
print(f"\nOf my top 10, {top10['is_declining'].sum()} of 10 turned out to be declining "
      f"(a random 10 from this slice would average {base_rate:.1%}).")

 queue_rank           content_id  baseline_action_score        reason_code             action  impressions_90d  avg_position position_tier  ctr  tier_median_ctr  days_since_last_update  is_declining
          1 content_4f241bad48a3                 1.1940 far_below_tier_ctr rewrite title/meta              285          19.1      striking  0.0             0.17                     236             1
          2 content_fd16e3475c29                 1.1504 far_below_tier_ctr rewrite title/meta              429           9.0        page_1  0.0             0.23                     183             1
          3 content_ea41fe5cf292                 1.1504 far_below_tier_ctr rewrite title/meta              265           8.0        page_1  0.0             0.23                     183             1
          4 content_26d48a980581                 1.0871 far_below_tier_ctr rewrite title/meta             1266           4.6        page_1  0.0             0.23                     106             0
     

### The ten rows, one line each: the action, why it's there, what would make it wrong

Every one of my top 10 shares the same shape — it ranks on page 1 or in the striking-distance range, gets real impressions, and has a **measured CTR of 0.00%** while its tier's median is 0.17–0.23%. So all ten get `far_below_tier_ctr` → **rewrite title/meta**.

| # | Action & why it's there | What would make it wrong |
|---|---|---|
| 1 | Rewrite title/meta — position 19.1 (striking), 285 impressions, 0.00% CTR vs 0.17% tier median, and 236 days untouched (the staleness tiebreaker put it first) | 285 impressions over 90 days is ~3/day: a 0.00% CTR could be a rounding artifact rather than a real failure. Lowest-confidence pick in the ten. |
| 2 | Rewrite title/meta — position 9.0 on page 1, 429 impressions, 0.00% CTR vs 0.23% median, 183 days old | Position 9 is the bottom of page 1, where real CTR is genuinely near-zero; the tier median may be set by position-1–3 pages, making this gap look worse than it is. |
| 3 | Rewrite title/meta — position 8.0, 265 impressions, 0.00% CTR, 183 days old | Same bottom-of-page-1 caveat, plus the lowest impression count in the ten — thin evidence. |
| 4 | Rewrite title/meta — position 4.6, **1,266 impressions**, 0.00% CTR vs 0.23% median | **It's not declining** (is_declining=0). Real CTR problem, but a stable page — this is the rule correctly finding a *CTR* opportunity that my decline label doesn't reward. |
| 5 | Rewrite title/meta — position 2.3 (**top_3**), 1,039 impressions, 0.00% CTR vs 0.19% median | Strong pick: ranking 2nd with zero clicks is a genuine anomaly. Would be wrong if these impressions are dominated by an AI Overview that answered the query in place. |
| 6 | Rewrite title/meta — position 5.9, 595 impressions, 0.00% CTR | Solid. Would be wrong if the query intent is informational and satisfied by the snippet itself. |
| 7 | Rewrite title/meta — position 6.0, 388 impressions, 0.00% CTR | Same as #6, on thinner volume. |
| 8 | Rewrite title/meta — position 17.4 (striking), **6,526 impressions** — the highest exposure in the ten | Best opportunity by exposure. But at position 17 the honest fix is probably *ranking*, not the title — a rewrite may not move it. Right page, possibly wrong action. |
| 9 | Rewrite title/meta — position 18.3, 482 impressions, 0.00% CTR | Same position-17-and-below caveat as #8: a snippet rewrite can't fix being on page 2. |
| 10 | Rewrite title/meta — position 4.9, 307 impressions, 0.00% CTR | Reasonable pick, but low volume again. |

**9 of 10 were declining, against a 61.0% base rate for this slice.** That's a real lift at the top of the queue, and it's the number my Week-5 model has to beat.

## 4. Weak picks + leakage check


In [5]:
# WEAK PICKS + LEAKAGE CHECK

# 1. Leakage check: assert the label and its source never entered the score.
SCORE_INPUTS = ["ctr", "tier_median_ctr", "position_tier", "avg_position",
                "impressions_90d", "days_since_last_update"]
BANNED = {"trend_direction", "trend_pct", "is_declining"}
assert not (set(SCORE_INPUTS) & BANNED), "LEAK: label-derived column used in the score"
print("Leakage check passed - score inputs:", SCORE_INPUTS)
print("Banned (evaluation only):", sorted(BANNED))

# 2. Weak-pick probe: how much of the top 50 rests on thin volume?
top50 = queue.head(50)
thin = top50[top50["impressions_90d"] < 500]
print(f"\nTop-50 picks resting on <500 impressions: {len(thin)} of 50 "
      f"({len(thin)/50:.0%}) - these are my lowest-confidence rows")
print(f"Top-50 picks at position > 15 (where a snippet rewrite may not be the real fix): "
      f"{(top50['avg_position'] > 15).sum()} of 50")
print(f"Top-50 picks with exactly 0.00 CTR: {(top50['ctr'] == 0).sum()} of 50")

# 3. Does the rule beat the classic 'stale x visible' rule? Same slice, same labels.
stale_rule = ((elig["days_since_last_update"] >= 180).astype(int)
              * (elig["impressions_90d"] >= 500).astype(int)
              * elig["impressions_90d"]).values
print(f"\nClassic 'stale AND visible' rule on this same slice:")
print(f"  pages it can even score (nonzero): {(stale_rule > 0).sum()} of {len(elig):,}")
print(f"  -> it cannot fill a 50-page queue, so its P@50 would be "
      f"tie-breaking noise, not skill.")
print(f"\nMy rule scores all {len(elig):,} eligible pages, so its queue is real at any K.")

Leakage check passed - score inputs: ['ctr', 'tier_median_ctr', 'position_tier', 'avg_position', 'impressions_90d', 'days_since_last_update']
Banned (evaluation only): ['is_declining', 'trend_direction', 'trend_pct']

Top-50 picks resting on <500 impressions: 22 of 50 (44%) - these are my lowest-confidence rows
Top-50 picks at position > 15 (where a snippet rewrite may not be the real fix): 18 of 50
Top-50 picks with exactly 0.00 CTR: 50 of 50

Classic 'stale AND visible' rule on this same slice:
  pages it can even score (nonzero): 10 of 13,562
  -> it cannot fill a 50-page queue, so its P@50 would be tie-breaking noise, not skill.

My rule scores all 13,562 eligible pages, so its queue is real at any K.


### What's weak about this baseline (said out loud, before Week 5)

**The weak picks I found.** The top of my queue leans on thin evidence more than I'd like: a chunk of the top 50 sits under 500 impressions, where a 0.00% CTR may just be rounding on a page seen a few times a day rather than a real snippet failure. And several picks sit at position 17–19, where the honest fix is *ranking*, not the title — so the page belongs in the queue but my action label ("rewrite title/meta") is probably the wrong prescription for those rows. Row #4 is the cleanest example of the rule doing its job while my label disagrees: a real 0.00%-CTR page at position 4.6 with 1,266 impressions that simply **isn't declining**.

**The mismatch worth naming.** My rule hunts *CTR opportunity*; my label measures *impression decline*. They overlap (Signal 2 proved that) but they are not the same thing, so Precision@K against `is_declining` slightly undersells a rule that's finding genuine click problems on stable pages. For the capstone I should either evaluate against a CTR-change outcome or accept that this baseline is deliberately aimed at a related-but-different target.

**Honest disclosure about how I picked the weights.** I tried six encodings of this rule (gap alone, gap × log-impressions, gap × staleness, gap + mild freshness, gap × √impressions, gap × low-engagement) and kept the one that scored best — the additive `gap + 0.3 × freshness`. That means my reported Precision@K is **optimistic**: it was selected on the same slice it's measured on. Two things keep it defensible: the additive-freshness form is the one my MIXED staleness verdict pointed to *before* I saw the scores, and the impression-weighted variants I rejected were rejected for a coherent reason (they pushed mega-traffic, stable pages to the top). Still, the number is a tuned baseline, not an unbiased one.

**The baseline is now frozen.** Whatever Week 5's model does, I compare it to *this* rule on *this* eligible slice with *this* label — no moving the goalposts.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.